# 08 — DeepLabCut SuperAnimal Top-Down Quality Check (Colab)

**Audience:** intern reviewing detection quality on new top-down mouse cage videos.

**What this notebook does**
1. Mounts your Google Drive and points at a single video.
2. Confirms the video is actually top-down (4-frame thumbnail check).
3. Runs a zero-shot DeepLabCut SuperAnimal pose model on the video.
4. Produces a per-keypoint reliability report (which keypoints DLC trusts).
5. Produces a multi-mouse identity diagnostic (does count stay stable? do trajectories swap?).
6. Renders a 30-second annotated MP4 you can scrub through.
7. Computes top-down body-shape features (body length, orientation, centroid speed, hull area).
8. Asks you a short checklist of yes/no questions to record findings.
9. Saves all artifacts back to Drive in a per-video folder.

**Why two model choices.** DeepLabCut publishes two relevant SuperAnimal models:
- `superanimal_topviewmouse` — trained on ≈ 5 000 lab mice filmed top-down. 27 keypoints, paw-aware. **Use this by default for top-down cage video.**
- `superanimal_quadruped` — trained on 40 000 quadrupeds (mice through elephants), mostly side-view. 39 keypoints. Useful as a comparison when you want to see whether the quadruped vocabulary buys anything extra.
Set `MODEL` in the configuration cell to switch.

**Runtime**
- Colab runtime: **GPU** (Runtime → Change runtime type → T4 minimum, A100 ideal).
- First-time model download: ≈ 1.5 GB cached under `~/.cache/torch`.
- Inference: ≈ 5–20 fps on T4, ≈ 30–60 fps on A100. A 17-minute 30 fps video = ≈ 30 000 frames → plan for 25–90 minutes of inference.

**Important.** This notebook is a *quality check*, not a behavior classifier. Don't hand-edit keypoints here — just record what works and what doesn't, then we decide downstream which keypoints to trust.

## 1. Environment setup
Install DeepLabCut (PyTorch backend, includes the model zoo), confirm we have a GPU, and mount Google Drive.

**What to look for:** the install cell finishes without red errors; `torch.cuda.is_available()` prints `True`; Drive mounts and you see your folders.

In [ ]:
# --- Dependency install (pinned, single resolver pass) -----------------------
# Why this is fussy: the numpy 1.x <-> 2.x ABI break. DeepLabCut (PyTorch
# backend) requires numpy<2 and downgrades it to 1.26.4. Anything COMPILED
# against numpy -- notably OpenCV -- must then be a numpy-1.x build, or
# `import cv2` crashes at runtime ("numpy.dtype size changed"). Installing
# opencv UNPINNED pulls the newest wheel (built on numpy 2) and re-creates the
# clash. So: pin numpy + a numpy-1.x OpenCV, install in ONE pass, then RESTART.
%pip install -q --pre deeplabcut
%pip install -q \
    "numpy==1.26.4" \
    "opencv-python-headless==4.10.0.84" \
    matplotlib pandas tqdm

# Colab pre-imports numpy 2.x at startup; the downgrade above only takes effect
# after a runtime restart. This kills the kernel on purpose -- once it comes
# back, just Run-all again (the install is cached, numpy stays pinned).
import os
os.kill(os.getpid(), 9)


In [ ]:
# --- Verify the environment is consistent (run AFTER the restart) ------------
# If this cell fails or the OpenCV import errors, the env is wrong -- do NOT
# proceed (that mismatch is what produced silent/black results before).
import numpy, cv2, deeplabcut, torch
print("numpy :", numpy.__version__)        # expect 1.26.x
print("opencv:", cv2.__version__)          # importing cleanly means ABI is OK
print("DLC   :", deeplabcut.__version__)
print("torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert numpy.__version__.startswith("1."), "numpy must be <2 for this DLC build"


In [ ]:
# Verify GPU. If this prints False, stop and switch the runtime to GPU.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Inference will be unusably slow. Runtime > Change runtime type > GPU.")

In [ ]:
# Mount Google Drive. After running, click the link, authenticate,
# paste the code back. Your Drive then lives at /content/drive/MyDrive/.
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration — the only cell you should edit
Set the video path and the knobs. Everything below reads from this cell.

In [ ]:
from pathlib import Path

# === REQUIRED ===
# Full path to the .MOV / .mp4 video on Drive. Upload one video at a time.
VIDEO_PATH = Path("/content/drive/MyDrive/mousercv/videos/Cage 17082 video.MOV")

# Where to write the per-video outputs (CSV, JSON, preview MP4, plots).
OUTPUT_ROOT = Path("/content/drive/MyDrive/mousercv/dlc_topdown_results")

# === MODEL ===
# 'superanimal_topviewmouse' (default, top-down lab mice, 27 keypoints) OR
# 'superanimal_quadruped'    (general quadruped, 39 keypoints, mostly side-view).
MODEL = "superanimal_topviewmouse"

# Detector + pose-model pair. These names are what video_inference_superanimal expects.
DETECTOR_NAME = "fasterrcnn_resnet50_fpn_v2"
MODEL_NAME = "hrnet_w32"

# How many mice you EXPECT in this cage. Used by the count-anomaly diagnostic.
EXPECTED_MICE = 2

# === RUNTIME KNOBS ===
# Process only the first N seconds. Set to None for full video.
# Recommended for first pass: 60 (one minute) so you can sanity-check fast.
MAX_SECONDS = 60

# Confidence threshold below which a keypoint is treated as 'not detected'
# in the reliability stats and the annotated preview.
PCUTOFF = 0.5

# Whether to run DLC's video adaptation pass. Improves accuracy by
# fine-tuning on the video itself; adds ~2x runtime. Leave False for
# the first quality check; turn on once the perspective is confirmed.
VIDEO_ADAPT = False

# === ANNOTATED PREVIEW ===
# Start time (seconds) for the 30-second annotated preview snippet.
PREVIEW_START_SEC = 10
PREVIEW_DURATION_SEC = 30

# === DERIVED ===
VIDEO_STEM = VIDEO_PATH.stem.replace(" ", "_")
OUTPUT_DIR = OUTPUT_ROOT / VIDEO_STEM
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/dlc_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Video:        ", VIDEO_PATH)
print("Exists:       ", VIDEO_PATH.exists())
print("Model:        ", MODEL, "/", MODEL_NAME, "+", DETECTOR_NAME)
print("Output dir:   ", OUTPUT_DIR)
print("Max seconds:  ", MAX_SECONDS)

## 3. Perspective sanity check
Pull 4 evenly-spaced frames out of the video and show them in a 2×2 grid.

**What to look for:** every frame should show the cage from above, with the floor of the cage filling most of the frame. If you see the side of the cage or an angled view, this is the wrong notebook — use `02_evaluate_superanimal.ipynb` for front/side angle. Stop and tell the team.

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = n_frames / fps if fps else 0
print(f"FPS: {fps:.2f}   Frames: {n_frames}   Resolution: {width}x{height}   Duration: {duration:.1f}s")

sample_idx = [int(n_frames * f) for f in (0.1, 0.4, 0.6, 0.9)]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, idx in zip(axes.flatten(), sample_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f"Frame {idx} — read failed"); ax.axis('off'); continue
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx}  (t = {idx/fps:.1f}s)")
    ax.axis('off')
cap.release()
fig.suptitle("Perspective check — confirm all frames show the cage from above", fontsize=14)
plt.tight_layout(); plt.show()

**INTERN CHECKPOINT.** Before running the inference cell, type your answer in the next cell:
- Does each frame show the cage clearly from above?
- How many mice are visible?
- Anything unusual (lighting changes, camera moves, cage lid in frame)?

In [ ]:
PERSPECTIVE_NOTES = """
Top-down? (yes / no):
Mice visible:
Other notes:
"""
print(PERSPECTIVE_NOTES)

## 4. (Optional) Trim the video to MAX_SECONDS
Skip this cell if you set `MAX_SECONDS = None`. Otherwise we copy the first N seconds to a small local file so DLC has less to chew on.

**Why ffmpeg-copy:** no re-encoding, takes seconds, the trimmed file is what we feed to DLC.

In [ ]:
import subprocess

if MAX_SECONDS is None:
    INFER_VIDEO = WORK_DIR / VIDEO_PATH.name
    if not INFER_VIDEO.exists():
        # Copy locally so DLC isn't streaming from Drive (much faster).
        import shutil
        shutil.copy(VIDEO_PATH, INFER_VIDEO)
    print("Using full video:", INFER_VIDEO)
else:
    INFER_VIDEO = WORK_DIR / f"{VIDEO_STEM}_first{MAX_SECONDS}s.mp4"
    if not INFER_VIDEO.exists():
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-ss", "0", "-t", str(MAX_SECONDS),
            "-i", str(VIDEO_PATH),
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "22",
            "-an", str(INFER_VIDEO),
        ]
        subprocess.run(cmd, check=True)
    print("Trimmed clip:", INFER_VIDEO, f"({INFER_VIDEO.stat().st_size/1e6:.1f} MB)")

## 5. Run SuperAnimal inference
This is the heavy step. First call downloads model weights (≈ 1.5 GB) and caches them; subsequent calls are fast to start. Inference itself runs at 5–60 fps depending on GPU.

**What to look for:** progress bar advances, no `CUDA out of memory`. If OOM, lower `batch_size` in the call below.

In [ ]:
import deeplabcut, time
print("DeepLabCut version:", deeplabcut.__version__)

t0 = time.time()
deeplabcut.video_inference_superanimal(
    [str(INFER_VIDEO)],
    superanimal_name=MODEL,
    model_name=MODEL_NAME,
    detector_name=DETECTOR_NAME,
    video_adapt=VIDEO_ADAPT,
    pcutoff=PCUTOFF,
    batch_size=8,
)
elapsed = time.time() - t0
print(f"Inference finished in {elapsed:.1f}s")

## 6. Locate and load the prediction files
DLC writes results next to the input video as `*_superanimal_*.h5` (and a matching `.json`). We find them, load, and flatten to a tidy long-form CSV: `(frame, individual, keypoint, x, y, confidence)`.

In [ ]:
import pandas as pd, numpy as np, glob

h5_candidates = sorted(WORK_DIR.glob(f"{INFER_VIDEO.stem}*{MODEL}*.h5"))
if not h5_candidates:
    h5_candidates = sorted(WORK_DIR.glob(f"{INFER_VIDEO.stem}*.h5"))
if not h5_candidates:
    raise FileNotFoundError(f"No DLC h5 next to {INFER_VIDEO}. Check the inference cell output.")

h5_path = h5_candidates[0]
print("Loading:", h5_path)
df = pd.read_hdf(h5_path)
print("Shape:", df.shape, "  Levels:", df.columns.names)
df.head(2)

In [ ]:
# Flatten the multi-index DataFrame into long form. Handles both single-animal
# (scorer, bodypart, coord) and multi-animal (scorer, individual, bodypart, coord) layouts.
scorer = df.columns.get_level_values(0)[0]
if df.columns.nlevels == 4:
    individuals = list(df.columns.get_level_values(1).unique())
    bodyparts   = list(df.columns.get_level_values(2).unique())
    multi_animal = True
else:
    individuals = ["animal"]
    bodyparts   = list(df.columns.get_level_values(1).unique())
    multi_animal = False

print(f"Multi-animal: {multi_animal}   Individuals: {individuals}   Bodyparts ({len(bodyparts)}): {bodyparts}")

rows = []
for frame_idx in range(len(df)):
    r = df.iloc[frame_idx]
    for ind in individuals:
        for bp in bodyparts:
            try:
                if multi_animal:
                    x = r[(scorer, ind, bp, "x")]; y = r[(scorer, ind, bp, "y")]; c = r[(scorer, ind, bp, "likelihood")]
                else:
                    x = r[(scorer, bp, "x")]; y = r[(scorer, bp, "y")]; c = r[(scorer, bp, "likelihood")]
            except KeyError:
                continue
            rows.append({"frame": frame_idx, "individual": ind, "keypoint": bp,
                         "x": float(x) if not np.isnan(x) else np.nan,
                         "y": float(y) if not np.isnan(y) else np.nan,
                         "confidence": float(c) if not np.isnan(c) else 0.0})
long_df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / f"{VIDEO_STEM}_keypoints.csv"
long_df.to_csv(csv_path, index=False)
print(f"Wrote {len(long_df):,} rows to {csv_path}")
long_df.head()

## 7. Pick the priority keypoints for behavior work
We tag 6 priority keypoints and check whether the model's vocabulary contains them. The two SuperAnimal models use slightly different names; we map both.

**What to look for:** `MISSING` in the printed table means that anatomical landmark just isn't in this model's vocabulary — you cannot use it for downstream behavior.

In [ ]:
# Anatomical-target -> per-model name mapping. Verified against
# DLClibrary superanimal_configs (supertopview.yaml has 27 kps; superquadruped.yaml has 39).
PRIORITY_TARGETS = ["nose", "front_left_paw", "front_right_paw",
                    "back_left_paw", "back_right_paw", "tail_base"]

MODEL_KP_MAP = {
    "superanimal_topviewmouse": {
        "nose": "nose",
        # TopViewMouse vocabulary lacks explicit paws; the closest joints are shoulders/hips.
        # We surface this clearly so the intern records it as a known limitation.
        "front_left_paw":  "left_shoulder",
        "front_right_paw": "right_shoulder",
        "back_left_paw":   "left_hip",
        "back_right_paw":  "right_hip",
        "tail_base": "tail_base",
    },
    "superanimal_quadruped": {
        "nose": "nose",
        "front_left_paw":  "front_left_paw",
        "front_right_paw": "front_right_paw",
        "back_left_paw":   "back_left_paw",
        "back_right_paw":  "back_right_paw",
        "tail_base": "tail_base",
    },
}

mapping = MODEL_KP_MAP[MODEL]
available = set(bodyparts)
priority_resolved = {}
print(f"{'TARGET':<20}{'MAPPED-TO':<25}{'STATUS'}")
print("-" * 60)
for tgt in PRIORITY_TARGETS:
    mapped = mapping.get(tgt)
    if mapped and mapped in available:
        priority_resolved[tgt] = mapped
        status = "OK (proxy)" if mapped != tgt else "OK"
    else:
        status = f"MISSING (looked for '{mapped}')"
    print(f"{tgt:<20}{str(mapped):<25}{status}")

## 8. Per-keypoint reliability over time
For each priority keypoint, plot confidence over time and report the % of frames above 0.3, 0.5, 0.7. A keypoint that lives below 0.5 most of the time is essentially unusable from this perspective.

**What to look for:** flat-low traces are bad (chronically occluded); spiky traces are okay (seen sometimes). Compare nose/tail (should be high from above) against paws (often occluded by body).

In [ ]:
import matplotlib.pyplot as plt

kp_summary = []
fig, axes = plt.subplots(len(priority_resolved), 1,
                          figsize=(14, 2.2 * max(1, len(priority_resolved))),
                          sharex=True, squeeze=False)
axes = axes[:, 0]

for ax, (target, kp) in zip(axes, priority_resolved.items()):
    sub = long_df[long_df["keypoint"] == kp]
    # Aggregate across individuals: take max confidence per frame (best detection wins).
    by_frame = sub.groupby("frame")["confidence"].max()
    ax.plot(by_frame.index / fps, by_frame.values, linewidth=0.7)
    ax.axhline(0.5, color="orange", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel(f"{target}\n({kp})", fontsize=8)

    pct = lambda thr: float((by_frame > thr).mean() * 100) if len(by_frame) else 0.0
    kp_summary.append({"target": target, "keypoint": kp,
                       "pct_above_0.3": round(pct(0.3), 1),
                       "pct_above_0.5": round(pct(0.5), 1),
                       "pct_above_0.7": round(pct(0.7), 1),
                       "mean_confidence": round(float(by_frame.mean()) if len(by_frame) else 0.0, 3)})

axes[-1].set_xlabel("time (s)")
fig.suptitle("Per-keypoint confidence over time (orange line = 0.5 cutoff)", fontsize=12)
plt.tight_layout(); plt.show()

kp_summary_df = pd.DataFrame(kp_summary)
print()
print(kp_summary_df.to_string(index=False))

## 9. Multi-mouse identity diagnostic
Plot the count of detected individuals per frame, and the centroid trajectories with one color per individual.

**What to look for:**
- Count line should sit near `EXPECTED_MICE` most of the time.
- Trajectories should look continuous — sudden teleports between two colors mean DLC swapped identities.

In [ ]:
if not multi_animal:
    print("Single-animal output — multi-mouse diagnostic skipped.")
else:
    # Count = number of individuals with at least one keypoint above PCUTOFF in the frame.
    detected = (long_df[long_df["confidence"] > PCUTOFF]
                .groupby(["frame", "individual"]).size().unstack(fill_value=0))
    counts_per_frame = (detected > 0).sum(axis=1)

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(counts_per_frame.index / fps, counts_per_frame.values, linewidth=0.6)
    ax.axhline(EXPECTED_MICE, color="red", linestyle="--", alpha=0.6, label=f"expected={EXPECTED_MICE}")
    ax.set_xlabel("time (s)"); ax.set_ylabel("# individuals detected")
    ax.set_title(f"Detected mouse count per frame (red = expected {EXPECTED_MICE})")
    ax.legend(); plt.tight_layout(); plt.show()

    n_anomalous = int((counts_per_frame != EXPECTED_MICE).sum())
    pct_anom = 100 * n_anomalous / len(counts_per_frame)
    print(f"Frames with count != {EXPECTED_MICE}: {n_anomalous}/{len(counts_per_frame)} ({pct_anom:.1f}%)")

In [ ]:
# Centroid trajectories per individual (one color per mouse).
if multi_animal:
    high = long_df[long_df["confidence"] > PCUTOFF]
    centroids = high.groupby(["frame", "individual"])[["x", "y"]].mean().reset_index()
    fig, ax = plt.subplots(figsize=(8, 8))
    for ind, sub in centroids.groupby("individual"):
        ax.plot(sub["x"], sub["y"], linewidth=0.6, alpha=0.7, label=ind)
    ax.invert_yaxis()  # image coords: y grows downward
    ax.set_aspect("equal")
    ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
    ax.set_title("Centroid trajectories — one color per individual")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout(); plt.show()

## 10. Annotated preview MP4 (30 seconds)
Render a snippet starting at `PREVIEW_START_SEC` with keypoints overlaid: nose/tail green, front-paw proxies blue, back-paw proxies red, other keypoints grey. Saved to Drive.

**What to look for:** open the MP4 and scrub through. Are keypoints sticking to the right anatomy? Any obvious identity swaps?

In [ ]:
import cv2, numpy as np
from tqdm import tqdm

preview_path = OUTPUT_DIR / f"{VIDEO_STEM}_preview_{PREVIEW_START_SEC}s_{PREVIEW_DURATION_SEC}s.mp4"

# Color scheme for the priority keypoints (BGR for OpenCV).
COLORS_BGR = {
    "nose": (0, 255, 0),               # green
    "tail_base": (0, 200, 0),          # green
    "front_left_paw": (255, 80, 0),    # blue
    "front_right_paw": (255, 80, 0),   # blue
    "back_left_paw": (0, 0, 255),      # red
    "back_right_paw": (0, 0, 255),     # red
}
INV_PRIORITY = {v: k for k, v in priority_resolved.items()}

# Per-individual outline color (for non-priority keypoints).
ind_palette = [(255, 255, 255), (255, 200, 0), (200, 0, 255), (0, 255, 255)]
ind_color = {ind: ind_palette[i % len(ind_palette)] for i, ind in enumerate(individuals)}

cap = cv2.VideoCapture(str(INFER_VIDEO))
src_fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
start_frame = int(PREVIEW_START_SEC * src_fps)
n_preview = int(PREVIEW_DURATION_SEC * src_fps)
end_frame = min(start_frame + n_preview, len(df))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(preview_path), fourcc, src_fps, (w, h))

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
for fidx in tqdm(range(start_frame, end_frame), desc="Rendering preview"):
    ok, frame = cap.read()
    if not ok: break
    sub = long_df[(long_df["frame"] == fidx) & (long_df["confidence"] > PCUTOFF)]
    for _, row in sub.iterrows():
        x, y = int(row["x"]), int(row["y"])
        kp = row["keypoint"]
        target = INV_PRIORITY.get(kp)
        if target and target in COLORS_BGR:
            cv2.circle(frame, (x, y), 6, COLORS_BGR[target], -1)
        else:
            cv2.circle(frame, (x, y), 3, ind_color.get(row["individual"], (180, 180, 180)), 1)
    cv2.putText(frame, f"f={fidx}  t={fidx/src_fps:.1f}s", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    writer.write(frame)
writer.release(); cap.release()
print("Saved preview:", preview_path)

## 11. Top-down body-shape features
From above, the things that matter behaviorally are: **body length** (nose → tail_base), **orientation** (nose-to-tail vector angle), **centroid speed** (px/s), and **keypoint-hull area** (rough body footprint).

These replace the mask-only features used in the front-angle pipeline.

**What to look for:** body length should be roughly constant for one mouse (with normal compression during grooming/rearing); centroid speed should spike when the mouse moves and rest at zero when still; hull area should stay in a stable range.

In [ ]:
from scipy.spatial import ConvexHull

def per_individual_features(ind_df, fps):
    feats = []
    for fidx, frame_df in ind_df.groupby("frame"):
        ok = frame_df[frame_df["confidence"] > PCUTOFF]
        record = {"frame": fidx, "t": fidx / fps,
                  "body_length": np.nan, "orientation_deg": np.nan,
                  "centroid_x": np.nan, "centroid_y": np.nan,
                  "hull_area": np.nan}
        nose = ok[ok["keypoint"] == priority_resolved.get("nose")]
        tail = ok[ok["keypoint"] == priority_resolved.get("tail_base")]
        if len(nose) and len(tail):
            nx, ny = nose.iloc[0]["x"], nose.iloc[0]["y"]
            tx, ty = tail.iloc[0]["x"], tail.iloc[0]["y"]
            record["body_length"] = float(np.hypot(nx - tx, ny - ty))
            record["orientation_deg"] = float(np.degrees(np.arctan2(ny - ty, nx - tx)))
        if len(ok):
            record["centroid_x"] = float(ok["x"].mean())
            record["centroid_y"] = float(ok["y"].mean())
        if len(ok) >= 3:
            pts = ok[["x", "y"]].dropna().values
            if len(pts) >= 3:
                try:
                    record["hull_area"] = float(ConvexHull(pts).volume)  # 2-D 'volume' = area
                except Exception:
                    pass
        feats.append(record)
    fdf = pd.DataFrame(feats).sort_values("frame").reset_index(drop=True)
    cx = fdf["centroid_x"].interpolate(); cy = fdf["centroid_y"].interpolate()
    fdf["speed_px_s"] = np.hypot(cx.diff(), cy.diff()) * fps
    return fdf

feature_frames = {}
for ind, ind_df in long_df.groupby("individual"):
    feature_frames[ind] = per_individual_features(ind_df, fps)

# Plot a 4-panel figure per individual.
for ind, fdf in feature_frames.items():
    fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(fdf["t"], fdf["body_length"], linewidth=0.7); axes[0].set_ylabel("body length (px)")
    axes[1].plot(fdf["t"], fdf["orientation_deg"], linewidth=0.7); axes[1].set_ylabel("orientation (deg)")
    axes[2].plot(fdf["t"], fdf["speed_px_s"], linewidth=0.7); axes[2].set_ylabel("speed (px/s)")
    axes[3].plot(fdf["t"], fdf["hull_area"], linewidth=0.7); axes[3].set_ylabel("hull area (px²)")
    axes[3].set_xlabel("time (s)")
    fig.suptitle(f"Top-down body-shape features — {ind}")
    plt.tight_layout(); plt.show()

    fdf.to_csv(OUTPUT_DIR / f"{VIDEO_STEM}_features_{ind}.csv", index=False)

## 12. Intern checklist
Fill in the answers below. Copy this filled cell into the shared Google Sheet for the video.

_The shared sheet template URL placeholder — replace with the real link before sharing with the intern: `https://docs.google.com/spreadsheets/d/REPLACE_ME/edit`_

In [ ]:
# Fill in the values, then re-run this cell. It writes a JSON next to the
# other artifacts and prints a copy-paste-friendly block for the sheet.
import json

CHECKLIST = {
    "video": str(VIDEO_PATH),
    "reviewer": "YOUR_NAME",
    "date": "YYYY-MM-DD",
    "perspective_is_topdown": None,                       # True / False
    "all_mice_detected_above_90pct": None,                # True / False
    "identity_swaps_observed": None,                      # True / False
    "unreliable_priority_keypoints": [],                  # list of names with <50% conf>0.3
    "nose_usable_topdown": None,                          # True / False
    "tailbase_usable_topdown": None,                      # True / False
    "paws_usable_topdown": None,                          # True / False / 'partially'
    "approx_inference_fps": None,                         # number
    "free_text_notes": "",
}

checklist_path = OUTPUT_DIR / f"{VIDEO_STEM}_intern_checklist.json"
with open(checklist_path, "w") as f:
    json.dump(CHECKLIST, f, indent=2)
print("Saved:", checklist_path)
print("\n--- copy below into the Google Sheet ---")
print(json.dumps(CHECKLIST, indent=2))

## 13. Export the reliability summary
Final step: write a small JSON with the per-keypoint stats so we can compare across videos later.

In [ ]:
import json

summary = {
    "video": str(VIDEO_PATH),
    "video_stem": VIDEO_STEM,
    "model": MODEL,
    "model_name": MODEL_NAME,
    "detector_name": DETECTOR_NAME,
    "video_adapt": VIDEO_ADAPT,
    "max_seconds": MAX_SECONDS,
    "pcutoff": PCUTOFF,
    "expected_mice": EXPECTED_MICE,
    "fps": float(fps),
    "frames_processed": int(len(df)),
    "individuals": individuals,
    "all_bodyparts": bodyparts,
    "priority_keypoint_resolution": priority_resolved,
    "per_keypoint_reliability": kp_summary,
    "artifacts": {
        "keypoints_csv": str(csv_path),
        "preview_mp4": str(preview_path),
        "intern_checklist_json": str(checklist_path),
    },
}
summary_path = OUTPUT_DIR / f"{VIDEO_STEM}_reliability_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("Wrote", summary_path)

print("\nAll artifacts in:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("  ", p.name, f"({p.stat().st_size/1e6:.2f} MB)")